# Experiment: Depth Effect on SAM/SAM2 Masks

Small feasibility experiment to test whether monocular depth can clean up or diagnose existing DINO+SAM2 masks.

This notebook does **not** modify the GUI or production pipeline. It does **not** rerun tracking, Qwen/VLM, or full 3D reconstruction. It starts from existing DINO+SAM2 proposals and compares original masks against conservative depth-refined masks and optional depth split candidates.

## 1. Setup and Config

In [ ]:
from __future__ import annotations

import json
import math
import random
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import torch
except Exception:
    torch = None

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "notebooks").exists() and (p / "outputs").exists():
            return p
        if (p / "scripts" / "auto_label").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def repo_path(value: str | Path | None) -> Path | None:
    if value is None or str(value).strip() == "":
        return None
    p = Path(value)
    return p if p.is_absolute() else REPO_ROOT / p


def auto_device() -> str:
    return "cuda" if torch is not None and torch.cuda.is_available() else "cpu"


CONFIG = {
    "INPUT_MODE": "existing_results",  # frames_dir, video, existing_results
    "FRAMES_DIR": "outputs/exp_2p5d_bytetrack_crop_extraction/frames",
    "VIDEO_PATH": "path/to/video.mp4",
    "EXISTING_DINO_SAM2_RESULTS": "outputs/exp_2p5d_bytetrack_crop_extraction/dino_sam2_raw/proposals_normalized.json",
    "OUTPUT_DIR": "outputs/exp_depth_effect_on_sam_masks",
    "SAMPLE_N": 12,
    "FRAME_STRIDE": 5,
    "RANDOM_SEED": 42,
    "USE_EXISTING_DINO_SAM2_RESULTS": True,
    "RUN_DINO_SAM2_IF_MISSING": False,
    "DINO_SAM2_PROMPT": None,
    "DINO_BOX_THRESHOLD": None,
    "DINO_TEXT_THRESHOLD": None,
    "DEPTH_MODEL": "depth_anything_v2",  # depth_anything_v2, midas, dummy
    "DEPTH_ANYTHING_REPO": r"C:\tmp\Depth-Anything-V2",
    "DEPTH_ANYTHING_ENCODER": "vits",
    "DEPTH_ANYTHING_CHECKPOINT": "models/depth_anything_v2_vits.pth",
    "DEPTH_INVERT": False,
    "DEPTH_CACHE": True,
    "DEVICE": auto_device(),
    "ENABLE_DEPTH_REFINEMENT": True,
    "DEPTH_KEEP_MODE": "adaptive_median",  # adaptive_median, percentile_band, kmeans_components
    "DEPTH_PERCENTILE_LOW": 15,
    "DEPTH_PERCENTILE_HIGH": 85,
    "DEPTH_MEDIAN_ABS_THRESHOLD": 0.12,
    "MIN_REFINED_AREA_RATIO": 0.35,
    "MAX_REFINED_AREA_RATIO": 1.10,
    "MIN_COMPONENT_AREA": 80,
    "ENABLE_DEPTH_SPLIT_CANDIDATE": True,
    "DEPTH_SPLIT_K": 2,
    "SPLIT_MIN_COMPONENT_RATIO": 0.20,
    "SPLIT_DEPTH_SEPARATION_THRESHOLD": 0.12,
    "SAVE_VIS": True,
    "SAVE_CONTACT_SHEET": True,
    "SHOW_INLINE": True,

    # Conservative depth policy
    "EDGE_ALIGNMENT_TRUST_THRESHOLD": 0.45,
    "BOUNDARY_TRIM_DEPTH_THRESHOLD": 0.14,
    "MIN_TRIM_AREA_RATIO": 0.75,
    "MAX_COMPONENT_INCREASE_AFTER_TRIM": 1,
    "MIN_BOUNDARY_OUTLIER_RATIO_TO_TRIM": 0.10,
    "HIGH_REFINE_RISK_THRESHOLD": 0.65,
    "SPLIT_REVIEW_SCORE_THRESHOLD": 0.60,
}

OUTPUT_DIR = repo_path(CONFIG["OUTPUT_DIR"])
assert OUTPUT_DIR is not None
DIRS = {
    "sampled_frames": OUTPUT_DIR / "sampled_frames",
    "depth": OUTPUT_DIR / "depth",
    "masks_original": OUTPUT_DIR / "masks_original",
    "masks_refined": OUTPUT_DIR / "masks_refined",
    "split_candidates": OUTPUT_DIR / "split_candidates",
    "visualizations": OUTPUT_DIR / "visualizations",
    "debug": OUTPUT_DIR / "debug",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("repo root:", REPO_ROOT)
print("device:", CONFIG["DEVICE"])
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, indent=2, ensure_ascii=False))

## 2. Load or Sample Frames

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def read_json_or_jsonl(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    return json.loads(path.read_text(encoding="utf-8"))


def image_size(path: Path) -> tuple[int, int]:
    img = cv2.imread(str(path))
    if img is None:
        return 0, 0
    h, w = img.shape[:2]
    return w, h


def sample_records(records: list[dict[str, Any]], n: int, seed: int) -> list[dict[str, Any]]:
    if len(records) <= n:
        return records
    rng = random.Random(seed)
    return sorted(rng.sample(records, n), key=lambda r: int(r.get("frame_idx", r.get("extracted_frame_idx", 0))))


def load_frames_from_dir(frames_dir: Path) -> pd.DataFrame:
    paths = sorted(p for p in frames_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    rows = []
    for i, p in enumerate(paths):
        w, h = image_size(p)
        rows.append({"frame_idx": i, "original_frame_idx": i, "frame_path": str(p), "width": w, "height": h})
    return pd.DataFrame(sample_records(rows, CONFIG["SAMPLE_N"], CONFIG["RANDOM_SEED"]))


def extract_sampled_video_frames(video_path: Path) -> pd.DataFrame:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    indices = list(range(0, total, max(1, int(CONFIG["FRAME_STRIDE"]))))
    if len(indices) > CONFIG["SAMPLE_N"]:
        indices = sorted(random.Random(CONFIG["RANDOM_SEED"]).sample(indices, CONFIG["SAMPLE_N"]))
    rows = []
    for out_idx, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            continue
        out = DIRS["sampled_frames"] / f"frame_{out_idx:06d}_orig_{frame_idx:08d}.jpg"
        cv2.imwrite(str(out), frame)
        h, w = frame.shape[:2]
        rows.append({"frame_idx": out_idx, "original_frame_idx": frame_idx, "timestamp_sec": frame_idx / fps if fps else None, "frame_path": str(out), "width": w, "height": h})
    cap.release()
    return pd.DataFrame(rows)


def frames_from_existing_results(path: Path) -> pd.DataFrame:
    data = read_json_or_jsonl(path)
    if isinstance(data, dict):
        data = data.get("frames") or data.get("results") or data.get("data") or []
    rows = []
    for i, rec in enumerate(data):
        frame_path = rec.get("frame_path") or rec.get("image_path")
        if not frame_path:
            continue
        p = repo_path(frame_path)
        if p is None or not p.exists():
            continue
        w, h = image_size(p)
        rows.append({
            "frame_idx": int(rec.get("frame_idx", rec.get("extracted_frame_idx", i))),
            "original_frame_idx": rec.get("original_frame_idx"),
            "frame_path": str(p),
            "width": w,
            "height": h,
        })
    return pd.DataFrame(sample_records(rows, CONFIG["SAMPLE_N"], CONFIG["RANDOM_SEED"]))


if CONFIG["INPUT_MODE"] == "frames_dir":
    frame_df = load_frames_from_dir(repo_path(CONFIG["FRAMES_DIR"]))
elif CONFIG["INPUT_MODE"] == "video":
    frame_df = extract_sampled_video_frames(repo_path(CONFIG["VIDEO_PATH"]))
elif CONFIG["INPUT_MODE"] == "existing_results":
    frame_df = frames_from_existing_results(repo_path(CONFIG["EXISTING_DINO_SAM2_RESULTS"]))
else:
    raise ValueError(f"Unknown INPUT_MODE: {CONFIG['INPUT_MODE']}")

print("sampled frames:", len(frame_df))
display(frame_df.head())

cols = min(4, len(frame_df))
rows = math.ceil(min(len(frame_df), 8) / cols) if cols else 1
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes:
    ax.axis("off")
for ax, (_, r) in zip(axes, frame_df.head(8).iterrows()):
    img = cv2.cvtColor(cv2.imread(r["frame_path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"frame {r['frame_idx']}")
plt.tight_layout(); plt.show()

## 3. Load Existing DINO/SAM2 Proposals or Run Placeholder

In [ ]:
def norm_bbox(value) -> list[float] | None:
    if value is None:
        return None
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except Exception:
            return None
    if isinstance(value, dict):
        if all(k in value for k in ["x1", "y1", "x2", "y2"]):
            return [float(value[k]) for k in ["x1", "y1", "x2", "y2"]]
    if isinstance(value, (list, tuple)) and len(value) >= 4:
        return [float(value[0]), float(value[1]), float(value[2]), float(value[3])]
    return None


def load_mask(mask_path: str | Path | None, shape_hw: tuple[int, int]) -> tuple[np.ndarray | None, str]:
    if mask_path:
        p = repo_path(mask_path)
        if p is not None and p.exists():
            m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if m is not None:
                if m.shape[:2] != shape_hw:
                    m = cv2.resize(m, (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_NEAREST)
                return (m > 0).astype(np.uint8), "sam2"
    return None, "missing"


def bbox_mask(bbox, shape_hw):
    h, w = shape_hw
    m = np.zeros((h, w), dtype=np.uint8)
    if bbox is None:
        return m
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 > x1 and y2 > y1:
        m[y1:y2, x1:x2] = 1
    return m


def normalize_detection(det: dict[str, Any], frame_rec: dict[str, Any], det_i: int) -> dict[str, Any] | None:
    bbox = norm_bbox(det.get("bbox") or det.get("bbox_xyxy") or det.get("box"))
    label = det.get("raw_label") or det.get("label") or det.get("class_name") or det.get("dino_label") or det.get("cluster_label") or "unknown"
    conf = det.get("conf", det.get("score", det.get("dino_conf", det.get("confidence", 0.0))))
    try:
        conf = float(conf)
    except Exception:
        conf = 0.0
    h, w = int(frame_rec["height"]), int(frame_rec["width"])
    mask, mask_source = load_mask(det.get("mask_path"), (h, w))
    if mask is None:
        mask = bbox_mask(bbox, (h, w))
        mask_source = "bbox_fallback"
    det_id = str(det.get("det_id", f"{int(frame_rec['frame_idx']):06d}_{det_i:04d}"))
    original_mask_path = DIRS["masks_original"] / f"frame_{int(frame_rec['frame_idx']):06d}_{det_id.replace('/', '_')}_orig.png"
    cv2.imwrite(str(original_mask_path), (mask * 255).astype(np.uint8))
    return {
        "det_id": det_id,
        "raw_label": str(label),
        "conf": conf,
        "bbox": bbox,
        "mask_path": str(original_mask_path),
        "crop_path": det.get("crop_path"),
        "mask_source": det.get("mask_source") or mask_source,
        "raw_metadata": det,
        "mask": mask,
    }


def load_existing_proposals(path: Path, frame_df: pd.DataFrame) -> list[dict[str, Any]]:
    data = read_json_or_jsonl(path)
    if isinstance(data, dict):
        data = data.get("frames") or data.get("results") or data.get("data") or []
    selected_paths = {str(Path(p).resolve()) for p in frame_df["frame_path"].tolist()}
    selected_idxs = {int(v) for v in frame_df["frame_idx"].tolist()}
    out = []
    for i, rec in enumerate(data):
        frame_path = rec.get("frame_path") or rec.get("image_path")
        p = repo_path(frame_path) if frame_path else None
        frame_idx = int(rec.get("frame_idx", rec.get("extracted_frame_idx", i)))
        path_match = p is not None and str(p.resolve()) in selected_paths
        if not path_match and frame_idx not in selected_idxs:
            continue
        row = frame_df[frame_df["frame_idx"] == frame_idx]
        if len(row) == 0 and p is not None:
            row = frame_df[frame_df["frame_path"].map(lambda x: str(Path(x).resolve())) == str(p.resolve())]
        if len(row) == 0:
            continue
        frame_rec = row.iloc[0].to_dict()
        norm_dets = []
        for j, det in enumerate(rec.get("detections") or rec.get("objects") or rec.get("proposals") or []):
            nd = normalize_detection(det, frame_rec, j)
            if nd is not None:
                norm_dets.append(nd)
        out.append({**frame_rec, "detections": norm_dets})
    return out


def run_dino_sam2_placeholder(frame_df: pd.DataFrame):
    raise NotImplementedError("Point EXISTING_DINO_SAM2_RESULTS to existing proposals for this notebook.")


if CONFIG["USE_EXISTING_DINO_SAM2_RESULTS"]:
    proposals_by_frame = load_existing_proposals(repo_path(CONFIG["EXISTING_DINO_SAM2_RESULTS"]), frame_df)
elif CONFIG["RUN_DINO_SAM2_IF_MISSING"]:
    proposals_by_frame = run_dino_sam2_placeholder(frame_df)
else:
    proposals_by_frame = [{**r.to_dict(), "detections": []} for _, r in frame_df.iterrows()]

flat_dets = [d for fr in proposals_by_frame for d in fr["detections"]]
print("frames with proposals:", len(proposals_by_frame))
print("detections:", len(flat_dets))
print("labels:", Counter(d["raw_label"] for d in flat_dets).most_common(20))
print("mask sources:", Counter(d["mask_source"] for d in flat_dets))


def overlay_masks(frame_rec, max_dets=20):
    img = cv2.imread(frame_rec["frame_path"])
    if img is None:
        return None
    out = img.copy()
    for k, d in enumerate(frame_rec["detections"][:max_dets]):
        color = ((37*k + 70) % 255, (83*k + 40) % 255, (131*k + 90) % 255)
        m = d["mask"].astype(bool)
        out[m] = (0.55 * out[m] + 0.45 * np.array(color)).astype(np.uint8)
        if d["bbox"]:
            x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            cv2.putText(out, f"{d['raw_label']} {d['conf']:.2f}", (x1, max(15, y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


show_n = min(3, len(proposals_by_frame))
if show_n:
    fig, axes = plt.subplots(show_n, 1, figsize=(8, 4 * show_n))
    axes = np.array(axes).reshape(-1)
    for ax, fr in zip(axes, proposals_by_frame[:show_n]):
        vis = overlay_masks(fr)
        if vis is None:
            vis = np.zeros((10, 10, 3), dtype=np.uint8)
        ax.imshow(vis)
        ax.set_title(f"frame {fr['frame_idx']} original masks ({len(fr['detections'])} dets)")
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 4. Depth Model Wrapper

In [ ]:
def normalize_depth(depth: np.ndarray) -> np.ndarray:
    depth = depth.astype(np.float32)
    finite = np.isfinite(depth)
    if not finite.any():
        return np.zeros_like(depth, dtype=np.float32)
    lo, hi = np.nanpercentile(depth[finite], [1, 99])
    if hi <= lo:
        lo, hi = float(np.nanmin(depth[finite])), float(np.nanmax(depth[finite]))
    if hi <= lo:
        return np.zeros_like(depth, dtype=np.float32)
    out = np.clip((depth - lo) / (hi - lo), 0, 1).astype(np.float32)
    if CONFIG["DEPTH_INVERT"]:
        out = 1.0 - out
    return out


class DepthWrapper:
    def __init__(self, config):
        self.config = config
        self.mode = config["DEPTH_MODEL"]
        self.device = config["DEVICE"]
        self.model = None
        self.transform = None
        self.available_mode = "dummy"
        if self.mode == "depth_anything_v2":
            self._try_depth_anything_v2()
        elif self.mode == "midas":
            self._try_midas()

    def _try_depth_anything_v2(self):
        try:
            repo = repo_path(self.config.get("DEPTH_ANYTHING_REPO"))
            if repo and repo.exists() and str(repo) not in sys.path:
                sys.path.insert(0, str(repo))
            from depth_anything_v2.dpt import DepthAnythingV2
            if torch is None:
                raise RuntimeError("torch unavailable")
            encoder = self.config.get("DEPTH_ANYTHING_ENCODER", "vits")
            model_configs = {
                "vits": {"encoder": "vits", "features": 64, "out_channels": [48, 96, 192, 384]},
                "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
                "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
            }
            ckpt = repo_path(self.config.get("DEPTH_ANYTHING_CHECKPOINT"))
            if ckpt is None or not ckpt.exists():
                raise FileNotFoundError(f"Depth Anything checkpoint not found: {ckpt}")
            self.model = DepthAnythingV2(**model_configs[encoder])
            self.model.load_state_dict(torch.load(str(ckpt), map_location="cpu"))
            self.model = self.model.to(self.device).eval()
            self.available_mode = f"depth_anything_v2_{encoder}"
            print("Depth Anything V2 ready:", ckpt)
        except Exception as exc:
            print("Depth Anything V2 unavailable, falling back to dummy:", exc)
            self.model = None
            self.available_mode = "dummy"

    def _try_midas(self):
        try:
            if torch is None:
                raise RuntimeError("torch unavailable")
            self.model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True).to(self.device).eval()
            self.transform = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True).small_transform
            self.available_mode = "midas_small"
            print("MiDaS ready")
        except Exception as exc:
            print("MiDaS unavailable, falling back to dummy:", exc)
            self.model = None
            self.available_mode = "dummy"

    def predict(self, frame_bgr: np.ndarray) -> np.ndarray:
        h, w = frame_bgr.shape[:2]
        if self.available_mode.startswith("depth_anything") and self.model is not None:
            with torch.inference_mode():
                depth = self.model.infer_image(frame_bgr)
            if depth.shape[:2] != (h, w):
                depth = cv2.resize(depth, (w, h), interpolation=cv2.INTER_CUBIC)
            return normalize_depth(depth)
        if self.available_mode.startswith("midas") and self.model is not None:
            rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            inp = self.transform(rgb).to(self.device)
            with torch.inference_mode():
                pred = self.model(inp)
                pred = torch.nn.functional.interpolate(pred.unsqueeze(1), size=(h, w), mode="bicubic", align_corners=False).squeeze()
            return normalize_depth(pred.detach().cpu().numpy())
        yy = np.linspace(0, 1, h, dtype=np.float32)[:, None]
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
        return normalize_depth(0.65 * (1 - yy) + 0.35 * gray)


def depth_key(frame_rec):
    return f"frame_{int(frame_rec['frame_idx']):06d}_{Path(frame_rec['frame_path']).stem}_{CONFIG['DEPTH_MODEL']}"


def get_depth_map(frame_bgr, frame_rec, wrapper):
    key = depth_key(frame_rec)
    npy = DIRS["depth"] / f"{key}.npy"
    png = DIRS["depth"] / f"{key}.png"
    if CONFIG["DEPTH_CACHE"] and npy.exists():
        return np.load(npy)
    depth = wrapper.predict(frame_bgr)
    np.save(npy, depth)
    vis = (plt.cm.viridis(depth)[:, :, :3] * 255).astype(np.uint8)
    cv2.imwrite(str(png), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
    return depth


depth_wrapper = DepthWrapper(CONFIG)
depth_maps = {}
for fr in tqdm(proposals_by_frame, desc="depth maps"):
    img = cv2.imread(fr["frame_path"])
    if img is None:
        continue
    depth_maps[int(fr["frame_idx"])] = get_depth_map(img, fr, depth_wrapper)

show_frames = proposals_by_frame[:min(5, len(proposals_by_frame))]
if show_frames:
    fig, axes = plt.subplots(len(show_frames), 2, figsize=(9, 3.5 * len(show_frames)))
    axes = np.array(axes).reshape(len(show_frames), 2)
    for row, fr in zip(axes, show_frames):
        img = cv2.cvtColor(cv2.imread(fr["frame_path"]), cv2.COLOR_BGR2RGB)
        depth = depth_maps[int(fr["frame_idx"])]
        row[0].imshow(img); row[0].set_title(f"RGB frame {fr['frame_idx']}"); row[0].axis("off")
        row[1].imshow(depth, cmap="viridis"); row[1].set_title("depth"); row[1].axis("off")
    plt.tight_layout(); plt.show()

## 5. Conservative Depth Mask Analysis

In [ ]:
def connected_component_stats(mask: np.ndarray, min_area: int = 1):
    mask = (mask > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    areas = [int(stats[i, cv2.CC_STAT_AREA]) for i in range(1, num) if int(stats[i, cv2.CC_STAT_AREA]) >= min_area]
    total = int(mask.sum())
    largest = max(areas) if areas else 0
    return len(areas), largest / max(1, total)


def bbox_area(bbox):
    if not bbox:
        return 0.0
    return max(0.0, float(bbox[2] - bbox[0]) * float(bbox[3] - bbox[1]))


def mask_boundary(mask: np.ndarray, k: int = 3) -> np.ndarray:
    mask = (mask > 0).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    grad = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, kernel)
    return (grad > 0).astype(np.uint8)


def depth_edges(depth: np.ndarray) -> np.ndarray:
    d = depth.astype(np.float32)
    sx = cv2.Sobel(d, cv2.CV_32F, 1, 0, ksize=3)
    sy = cv2.Sobel(d, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(sx * sx + sy * sy)
    hi = float(np.percentile(mag[np.isfinite(mag)], 99)) if np.isfinite(mag).any() else 0.0
    if hi <= 1e-8:
        return np.zeros_like(d, dtype=np.float32)
    return np.clip(mag / hi, 0, 1).astype(np.float32)


def ring_mask(mask, radius=9):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (radius, radius))
    dil = cv2.dilate((mask > 0).astype(np.uint8), kernel, iterations=1)
    return ((dil > 0) & ~(mask.astype(bool))).astype(np.uint8)


def score_depth_compactness(depth_std, depth_range):
    return float(np.clip(1.0 - (0.55 * depth_std / 0.18 + 0.45 * depth_range / 0.35), 0, 1))


def score_fg_bg_separation(sep):
    return float(np.clip((sep if np.isfinite(sep) else 0.0) / 0.18, 0, 1))


def boundary_edge_alignment(mask: np.ndarray, edge: np.ndarray) -> float:
    b = mask_boundary(mask, 3)
    if b.sum() == 0:
        return 0.0
    near = cv2.dilate(b, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1).astype(bool)
    return float(np.clip(np.mean(edge[near]), 0, 1))


def base_mask_depth_metrics(mask: np.ndarray, depth: np.ndarray, bbox=None) -> dict[str, Any]:
    mask = (mask > 0).astype(np.uint8)
    vals = depth[mask.astype(bool)]
    area = int(mask.sum())
    comp_count, largest_ratio = connected_component_stats(mask, CONFIG["MIN_COMPONENT_AREA"])
    bbox_a = bbox_area(bbox)
    if vals.size == 0:
        stats = {k: np.nan for k in ["depth_median", "depth_mean", "depth_std", "depth_p10", "depth_p20", "depth_p80", "depth_p90"]}
    else:
        stats = {
            "depth_median": float(np.median(vals)),
            "depth_mean": float(np.mean(vals)),
            "depth_std": float(np.std(vals)),
            "depth_p10": float(np.percentile(vals, 10)),
            "depth_p20": float(np.percentile(vals, 20)),
            "depth_p80": float(np.percentile(vals, 80)),
            "depth_p90": float(np.percentile(vals, 90)),
        }
    depth_range_80 = float(stats["depth_p80"] - stats["depth_p20"]) if vals.size else np.nan
    depth_range_90 = float(stats["depth_p90"] - stats["depth_p10"]) if vals.size else np.nan
    ring = ring_mask(mask)
    ring_vals = depth[ring.astype(bool)]
    ring_median = float(np.median(ring_vals)) if ring_vals.size else np.nan
    fg_bg_sep = float(abs(stats["depth_median"] - ring_median)) if vals.size and np.isfinite(ring_median) else np.nan
    compact = score_depth_compactness(float(stats["depth_std"] or 0), depth_range_80 if np.isfinite(depth_range_80) else 1)
    fill = area / max(1.0, bbox_a)
    return {
        "mask_area": area,
        "bbox_area": bbox_a,
        "mask_bbox_fill_ratio": fill,
        "connected_component_count": comp_count,
        "largest_component_ratio": largest_ratio,
        **stats,
        "depth_range_p80_p20": depth_range_80,
        "depth_range_p90_p10": depth_range_90,
        "ring_depth_median": ring_median,
        "fg_bg_depth_separation": fg_bg_sep,
        "depth_compactness_score": compact,
        "foreground_background_separation_score": score_fg_bg_separation(fg_bg_sep),
    }


metric_rows = []
for fr in proposals_by_frame:
    depth = depth_maps.get(int(fr["frame_idx"]))
    if depth is None:
        continue
    edge = depth_edges(depth)
    for d in fr["detections"]:
        met = base_mask_depth_metrics(d["mask"], depth, d.get("bbox"))
        align = boundary_edge_alignment(d["mask"], edge)
        d["depth_edges"] = edge
        d["metrics_before"] = {**met, "depth_edge_alignment": align, "depth_edge_alignment_score": align}
        metric_rows.append({
            "frame_idx": int(fr["frame_idx"]),
            "frame_path": fr["frame_path"],
            "det_id": d["det_id"],
            "label": d["raw_label"],
            "conf": d["conf"],
            "mask_source": d["mask_source"],
            **d["metrics_before"],
        })
metrics_df = pd.DataFrame(metric_rows)
metrics_path = DIRS["debug"] / "depth_mask_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("saved:", metrics_path)
display(metrics_df.sort_values(["depth_edge_alignment", "depth_compactness_score"], ascending=[True, True]).head(20))

## 6. Conservative Boundary-only Trim Candidate and Action Policy

In [ ]:
def clean_components(mask: np.ndarray, min_area: int) -> np.ndarray:
    mask = (mask > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    out = np.zeros_like(mask)
    for i in range(1, num):
        if int(stats[i, cv2.CC_STAT_AREA]) >= min_area:
            out[labels == i] = 1
    return out


def component_bbox(mask: np.ndarray):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)]


def depth_split_candidate_info(mask: np.ndarray, depth: np.ndarray, config: dict[str, Any]):
    original = (mask > 0).astype(np.uint8)
    vals = depth[original.astype(bool)]
    if not config["ENABLE_DEPTH_SPLIT_CANDIDATE"] or vals.size < config["MIN_COMPONENT_AREA"] * 2:
        return {"split_candidate": False, "split_candidate_score": 0.0, "reason": "disabled_or_too_small", "component_masks": []}
    try:
        from sklearn.cluster import KMeans
        coords = np.column_stack(np.where(original.astype(bool)))
        km = KMeans(n_clusters=config["DEPTH_SPLIT_K"], random_state=config["RANDOM_SEED"], n_init=5).fit(vals.reshape(-1, 1))
        labels = km.labels_
        comps = []
        for k in range(config["DEPTH_SPLIT_K"]):
            comp = np.zeros_like(original)
            pts = coords[labels == k]
            if len(pts) == 0:
                continue
            comp[pts[:, 0], pts[:, 1]] = 1
            comp = clean_components(comp, config["MIN_COMPONENT_AREA"])
            area = int(comp.sum())
            if area:
                comps.append({"mask": comp, "area": area, "depth_median": float(np.median(depth[comp.astype(bool)])), "bbox": component_bbox(comp)})
    except Exception:
        med = float(np.median(vals))
        comps = []
        for comp in [((original > 0) & (depth <= med)).astype(np.uint8), ((original > 0) & (depth > med)).astype(np.uint8)]:
            comp = clean_components(comp, config["MIN_COMPONENT_AREA"])
            area = int(comp.sum())
            if area:
                comps.append({"mask": comp, "area": area, "depth_median": float(np.median(depth[comp.astype(bool)])), "bbox": component_bbox(comp)})
    total = max(1, int(original.sum()))
    comps = sorted(comps, key=lambda c: c["area"], reverse=True)
    large = [c for c in comps if c["area"] / total >= config["SPLIT_MIN_COMPONENT_RATIO"]]
    if len(large) < 2:
        return {"split_candidate": False, "split_candidate_score": 0.0, "num_components": len(comps), "reason": "not_enough_large_components", "component_masks": [c["mask"] for c in comps[:2]]}
    sep = abs(float(large[0]["depth_median"] - large[1]["depth_median"]))
    large_balance = min(large[0]["area"], large[1]["area"]) / max(1, max(large[0]["area"], large[1]["area"]))
    sep_score = float(np.clip(sep / max(1e-6, config["SPLIT_DEPTH_SEPARATION_THRESHOLD"] * 1.5), 0, 1))
    split_score = float(np.clip(0.75 * sep_score + 0.25 * large_balance, 0, 1))
    split = sep >= config["SPLIT_DEPTH_SEPARATION_THRESHOLD"] and split_score >= config["SPLIT_REVIEW_SCORE_THRESHOLD"]
    return {
        "split_candidate": bool(split),
        "split_candidate_score": split_score,
        "num_components": len(comps),
        "component_depths": [c["depth_median"] for c in comps],
        "component_areas": [c["area"] for c in comps],
        "component_bboxes": [c["bbox"] for c in comps],
        "depth_separation": sep,
        "reason": "large_depth_separated_components" if split else "depth_separation_or_score_too_small",
        "component_masks": [c["mask"] for c in comps[:2]],
    }


def conservative_boundary_trim_candidate(mask: np.ndarray, depth: np.ndarray, config: dict[str, Any]):
    original = (mask > 0).astype(np.uint8)
    if original.sum() == 0:
        return original, {"trim_valid": False, "reason": "empty_mask", "boundary_outlier_ratio": 0.0, "area_ratio_after_trim": 1.0}
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    core = cv2.erode(original, kernel, iterations=1)
    boundary = ((original > 0) & ~(core > 0)).astype(np.uint8)
    core_vals = depth[core.astype(bool)]
    if core_vals.size < config["MIN_COMPONENT_AREA"]:
        # If erosion destroys the object, use original depth stats and still only trim boundary.
        core_vals = depth[original.astype(bool)]
    if core_vals.size == 0 or boundary.sum() == 0:
        return original, {"trim_valid": False, "reason": "no_core_or_boundary", "boundary_outlier_ratio": 0.0, "area_ratio_after_trim": 1.0}
    med = float(np.median(core_vals))
    p20, p80 = np.percentile(core_vals, [20, 80])
    thr = max(float(config["BOUNDARY_TRIM_DEPTH_THRESHOLD"]), float(p80 - p20) * 1.25)
    boundary_outlier = ((boundary > 0) & (np.abs(depth - med) > thr)).astype(np.uint8)
    cleaned_boundary = ((boundary > 0) & ~(boundary_outlier > 0)).astype(np.uint8)
    candidate = ((core > 0) | (cleaned_boundary > 0)).astype(np.uint8)
    candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)
    orig_area = int(original.sum())
    cand_area = int(candidate.sum())
    area_ratio = cand_area / max(1, orig_area)
    orig_comps, _ = connected_component_stats(original, config["MIN_COMPONENT_AREA"])
    cand_comps, _ = connected_component_stats(candidate, config["MIN_COMPONENT_AREA"])
    outlier_ratio = int(boundary_outlier.sum()) / max(1, int(boundary.sum()))
    valid = (
        area_ratio >= config["MIN_TRIM_AREA_RATIO"]
        and cand_comps <= orig_comps + config["MAX_COMPONENT_INCREASE_AFTER_TRIM"]
        and outlier_ratio >= config["MIN_BOUNDARY_OUTLIER_RATIO_TO_TRIM"]
    )
    return candidate, {
        "trim_valid": bool(valid),
        "core_area": int(core.sum()),
        "boundary_area": int(boundary.sum()),
        "boundary_outlier_area": int(boundary_outlier.sum()),
        "boundary_outlier_ratio": float(outlier_ratio),
        "area_ratio_after_trim": float(area_ratio),
        "orig_components": int(orig_comps),
        "trim_components": int(cand_comps),
        "core_depth_median": med,
        "boundary_trim_threshold": float(thr),
        "removed_boundary_mask": boundary_outlier,
        "reason": "trim_passed" if valid else "trim_failed_safety_checks",
    }


def decide_conservative_action(metrics, trim_meta, split_info):
    edge_align = float(metrics.get("depth_edge_alignment", 0.0))
    compact = float(metrics.get("depth_compactness_score", 0.0))
    sep_score = float(metrics.get("foreground_background_separation_score", 0.0))
    outlier_ratio = float(trim_meta.get("boundary_outlier_ratio", 0.0))
    split_score = float(split_info.get("split_candidate_score", 0.0))
    refine_risk = float(np.clip(
        0.30 * (1.0 - compact)
        + 0.20 * (1.0 - sep_score)
        + 0.20 * (1.0 - edge_align)
        + 0.15 * outlier_ratio
        + 0.15 * split_score,
        0,
        1,
    ))
    notes = []
    replace_mask = False
    action = "accept_original"

    if split_info.get("split_candidate"):
        action = "split_candidate_review"
        notes.append("two large depth layers; keep original and review")
    elif edge_align >= CONFIG["EDGE_ALIGNMENT_TRUST_THRESHOLD"]:
        action = "accept_original"
        notes.append("SAM2 boundary aligns with depth edge")
    elif trim_meta.get("trim_valid") and refine_risk < CONFIG["HIGH_REFINE_RISK_THRESHOLD"]:
        action = "trim_boundary_noise"
        replace_mask = True
        notes.append("only boundary depth outliers removed; core preserved")
    elif refine_risk >= CONFIG["HIGH_REFINE_RISK_THRESHOLD"]:
        action = "reject_or_review" if compact < 0.20 and sep_score < 0.20 else "keep_original_review"
        notes.append("high depth risk; keep original for review")
    else:
        action = "accept_original"
        notes.append("no safe depth edit needed")

    return action, replace_mask, refine_risk, "; ".join(notes)


analysis_records = []
split_rows = []
for fr in tqdm(proposals_by_frame, desc="conservative depth analysis"):
    depth = depth_maps.get(int(fr["frame_idx"]))
    if depth is None:
        continue
    for d in fr["detections"]:
        metrics = d.get("metrics_before") or base_mask_depth_metrics(d["mask"], depth, d.get("bbox"))
        trim_mask, trim_meta = conservative_boundary_trim_candidate(d["mask"], depth, CONFIG)
        split_info = depth_split_candidate_info(d["mask"], depth, CONFIG)
        comp_masks = split_info.pop("component_masks", [])
        if split_info.get("split_candidate"):
            for ci, comp in enumerate(comp_masks):
                out = DIRS["split_candidates"] / f"frame_{int(fr['frame_idx']):06d}_{d['det_id'].replace('/', '_')}_comp{ci}.png"
                cv2.imwrite(str(out), (comp * 255).astype(np.uint8))
        action, replace_mask, risk, notes = decide_conservative_action(metrics, trim_meta, split_info)
        final_mask = trim_mask if replace_mask and action == "trim_boundary_noise" else d["mask"]
        out_path = DIRS["masks_refined"] / f"frame_{int(fr['frame_idx']):06d}_{d['det_id'].replace('/', '_')}_final.png"
        cv2.imwrite(str(out_path), (final_mask * 255).astype(np.uint8))
        removed = trim_meta.get("removed_boundary_mask")
        if isinstance(removed, np.ndarray) and int(removed.sum()) > 0:
            rem_path = DIRS["masks_refined"] / f"frame_{int(fr['frame_idx']):06d}_{d['det_id'].replace('/', '_')}_removed_boundary.png"
            cv2.imwrite(str(rem_path), (removed * 255).astype(np.uint8))
        d.update({
            "trim_candidate_mask": trim_mask,
            "final_mask": final_mask,
            "refined_mask": final_mask,  # backward-compatible name for later cells
            "final_mask_path": str(out_path),
            "refined_mask_path": str(out_path),
            "removed_boundary_mask": removed if isinstance(removed, np.ndarray) else np.zeros_like(d["mask"]),
            "action": action,
            "replace_mask": bool(replace_mask),
            "refine_action": action,
            "trim_metadata": {k: v for k, v in trim_meta.items() if not isinstance(v, np.ndarray)},
            "depth_split_info": split_info,
            "refine_risk_score": risk,
            "notes": notes,
            "metrics_after": base_mask_depth_metrics(final_mask, depth, d.get("bbox")),
        })
        row = {
            "frame_idx": int(fr["frame_idx"]),
            "det_id": d["det_id"],
            "label": d["raw_label"],
            "conf": d["conf"],
            "mask_source": d["mask_source"],
            "action": action,
            "replace_mask": bool(replace_mask),
            "depth_edge_alignment": metrics.get("depth_edge_alignment", 0.0),
            "depth_edge_alignment_score": metrics.get("depth_edge_alignment_score", metrics.get("depth_edge_alignment", 0.0)),
            "depth_compactness_score": metrics.get("depth_compactness_score", 0.0),
            "foreground_background_separation_score": metrics.get("foreground_background_separation_score", 0.0),
            "boundary_outlier_ratio": trim_meta.get("boundary_outlier_ratio", 0.0),
            "split_candidate": bool(split_info.get("split_candidate")),
            "split_candidate_score": split_info.get("split_candidate_score", 0.0),
            "area_ratio_after_trim": trim_meta.get("area_ratio_after_trim", 1.0),
            "refine_risk_score": risk,
            "notes": notes,
        }
        analysis_records.append(row)
        split_rows.append({"frame_idx": int(fr["frame_idx"]), "det_id": d["det_id"], "label": d["raw_label"], **split_info})

analysis_df = pd.DataFrame(analysis_records)
refined_df = analysis_df.copy()  # backward-compatible variable name
split_df = pd.DataFrame(split_rows)
analysis_path = DIRS["debug"] / "conservative_depth_mask_actions.csv"
analysis_df.to_csv(analysis_path, index=False)
print("saved:", analysis_path)
display(analysis_df.sort_values(["replace_mask", "split_candidate", "refine_risk_score"], ascending=[False, False, False]).head(30))

## 7. Depth Component Split Candidate Review

In [ ]:
# Split candidates are review signals only. They never replace original masks in this notebook.
if len(split_df):
    split_review_df = split_df.sort_values("split_candidate_score" if "split_candidate_score" in split_df else "split_candidate", ascending=False)
    display(split_review_df.head(30))
    split_review_df.to_csv(DIRS["debug"] / "depth_split_candidates.csv", index=False)
else:
    print("No split candidate rows.")

## 8. Visualization: Original vs Conservative Final Mask

In [ ]:
def draw_overlay(frame_bgr, detections, mask_key="mask", show_action=False):
    out = frame_bgr.copy()
    for i, d in enumerate(detections):
        color = np.array(((41*i + 80) % 255, (97*i + 30) % 255, (151*i + 50) % 255), dtype=np.uint8)
        m = d.get(mask_key)
        if m is None:
            continue
        sel = m.astype(bool)
        out[sel] = (0.55 * out[sel] + 0.45 * color).astype(np.uint8)
        if d.get("bbox"):
            x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
            cv2.rectangle(out, (x1, y1), (x2, y2), tuple(int(v) for v in color), 2)
            label = d["raw_label"]
            if show_action:
                label = f"{label} | {d.get('action', '')} r{d.get('refine_risk_score', 0):.2f}"
            cv2.putText(out, label[:70], (x1, max(16, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, tuple(int(v) for v in color), 1)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


def boundary_edge_panel(frame_bgr, depth, detections):
    edge = depth_edges(depth)
    rgb = cv2.cvtColor(frame_bgr.copy(), cv2.COLOR_BGR2RGB)
    edge_rgb = (plt.cm.magma(edge)[:, :, :3] * 255).astype(np.uint8)
    out = (0.45 * rgb + 0.55 * edge_rgb).astype(np.uint8)
    for d in detections:
        b = mask_boundary(d["mask"], 3).astype(bool)
        out[b] = [40, 240, 70]
    return out


def removed_boundary_panel(frame_bgr, detections):
    rgb = cv2.cvtColor(frame_bgr.copy(), cv2.COLOR_BGR2RGB)
    out = rgb.copy()
    overlay = np.zeros_like(out)
    any_removed = False
    for d in detections:
        rem = d.get("removed_boundary_mask")
        if isinstance(rem, np.ndarray) and int(rem.sum()) > 0:
            any_removed = True
            overlay[rem.astype(bool)] = [255, 40, 40]
    if any_removed:
        out = (0.65 * out + 0.35 * overlay).astype(np.uint8)
    return out


def action_panel(detections, width=900, height=520):
    canvas = np.full((height, width, 3), 255, dtype=np.uint8)
    y = 28
    cv2.putText(canvas, "Action labels and risk metrics", (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (20,20,20), 2)
    y += 28
    for d in detections[:18]:
        text = (
            f"{d['det_id']} {d['raw_label']} conf={d['conf']:.2f} | {d.get('action')} | "
            f"replace={d.get('replace_mask')} edge={d.get('metrics_before',{}).get('depth_edge_alignment',0):.2f} "
            f"out={d.get('trim_metadata',{}).get('boundary_outlier_ratio',0):.2f} "
            f"split={d.get('depth_split_info',{}).get('split_candidate', False)} "
            f"risk={d.get('refine_risk_score',0):.2f}"
        )
        cv2.putText(canvas, text[:130], (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (30,30,30), 1)
        y += 24
    return canvas


def visualize_frame(fr):
    frame = cv2.imread(fr["frame_path"])
    if frame is None:
        return None
    depth = depth_maps[int(fr["frame_idx"])]
    dets = fr["detections"]
    fig, axes = plt.subplots(2, 3, figsize=(22, 11))
    axes = axes.ravel()
    axes[0].imshow(draw_overlay(frame, dets, "mask")); axes[0].set_title("A. RGB + original SAM2 masks"); axes[0].axis("off")
    axes[1].imshow(depth, cmap="viridis"); axes[1].set_title("B. Depth map"); axes[1].axis("off")
    axes[2].imshow(boundary_edge_panel(frame, depth, dets)); axes[2].set_title("C. Depth edges + mask boundary"); axes[2].axis("off")
    axes[3].imshow(draw_overlay(frame, dets, "final_mask", show_action=True)); axes[3].set_title("D. Conservative final mask"); axes[3].axis("off")
    axes[4].imshow(removed_boundary_panel(frame, dets)); axes[4].set_title("E. Removed boundary pixels only"); axes[4].axis("off")
    axes[5].imshow(action_panel(dets)); axes[5].set_title("F. Actions / risk metrics"); axes[5].axis("off")
    fig.suptitle(f"frame {fr['frame_idx']} | dets {len(dets)}")
    plt.tight_layout()
    out = DIRS["visualizations"] / f"frame_{int(fr['frame_idx']):06d}_conservative_depth_policy.png"
    if CONFIG["SAVE_VIS"]:
        fig.savefig(out, dpi=150)
    if CONFIG["SHOW_INLINE"]:
        plt.show()
    else:
        plt.close(fig)
    return out


vis_paths = [p for p in (visualize_frame(fr) for fr in proposals_by_frame) if p]


def make_contact_sheet(paths: list[Path], out_path: Path, thumb_w=480):
    imgs = []
    for p in paths:
        img = cv2.imread(str(p))
        if img is None:
            continue
        h, w = img.shape[:2]
        scale = thumb_w / max(1, w)
        imgs.append(cv2.resize(img, (thumb_w, int(h * scale)), interpolation=cv2.INTER_AREA))
    if not imgs:
        return None
    cols = 2
    rows = math.ceil(len(imgs) / cols)
    thumb_h = max(im.shape[0] for im in imgs)
    canvas = np.full((rows * thumb_h, cols * thumb_w, 3), 245, dtype=np.uint8)
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        y, x = r * thumb_h, c * thumb_w
        canvas[y:y+im.shape[0], x:x+im.shape[1]] = im
    cv2.imwrite(str(out_path), canvas)
    return out_path


if CONFIG["SAVE_CONTACT_SHEET"]:
    sheet = make_contact_sheet(vis_paths, DIRS["visualizations"] / "conservative_depth_policy_contact_sheet.jpg")
    print("contact sheet:", sheet)

## 9. Conservative Action Summary

In [ ]:
summary_df = analysis_df[[
    "frame_idx", "det_id", "label", "action", "replace_mask", "depth_edge_alignment",
    "boundary_outlier_ratio", "split_candidate", "area_ratio_after_trim",
    "refine_risk_score", "notes"
]].copy()
summary_path = DIRS["debug"] / "depth_refinement_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary = {
    "total_detections": int(len(summary_df)),
    "accept_original_count": int((summary_df["action"] == "accept_original").sum()),
    "trim_boundary_noise_count": int((summary_df["action"] == "trim_boundary_noise").sum()),
    "split_candidate_review_count": int((summary_df["action"] == "split_candidate_review").sum()),
    "keep_original_review_count": int((summary_df["action"] == "keep_original_review").sum()),
    "reject_or_review_count": int((summary_df["action"] == "reject_or_review").sum()),
    "replace_mask_count": int(summary_df["replace_mask"].sum()),
    "average_depth_edge_alignment": float(summary_df["depth_edge_alignment"].mean()) if len(summary_df) else 0,
    "average_boundary_outlier_ratio": float(summary_df["boundary_outlier_ratio"].mean()) if len(summary_df) else 0,
    "average_refine_risk_score": float(summary_df["refine_risk_score"].mean()) if len(summary_df) else 0,
    "labels_most_often_trimmed": analysis_df[analysis_df["action"] == "trim_boundary_noise"]["label"].value_counts().head(10).to_dict(),
    "labels_most_often_split_review": analysis_df[analysis_df["action"] == "split_candidate_review"]["label"].value_counts().head(10).to_dict(),
}
(DIRS["debug"] / "depth_refinement_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, indent=2, default=str))
display(summary_df.sort_values(["replace_mask", "split_candidate", "refine_risk_score"], ascending=[False, False, False]).head(40))

## 10. Manual Feasibility Decision Cell

In [ ]:
total = max(1, len(summary_df))
trim_ratio = (summary_df["action"] == "trim_boundary_noise").mean() if total else 0
review_ratio = summary_df["action"].isin(["split_candidate_review", "keep_original_review", "reject_or_review"]).mean() if total else 0
replace_ratio = summary_df["replace_mask"].mean() if total else 0
avg_risk = float(summary_df["refine_risk_score"].mean()) if len(summary_df) else 0

if trim_ratio > 0.03 and replace_ratio < 0.25 and avg_risk < 0.55:
    decision = "Feasibility: promising. Depth is useful for conservative boundary-noise trimming and review signals."
elif review_ratio > 0.15 or trim_ratio > 0:
    decision = "Feasibility: mixed. Use depth mostly as a review/risk signal; keep original masks by default."
else:
    decision = "Feasibility: weak for mask modification. Use depth as scoring/review only."

print(decision)
print()
print("Policy reminder:")
print("- Original SAM2 mask is final unless action == trim_boundary_noise and safety checks pass.")
print("- High depth edge alignment means the SAM2 boundary is trusted.")
print("- Split candidates are review-only and never auto-split here.")
print()
print("Tuning suggestions:")
print("- If good masks are still trimmed: raise EDGE_ALIGNMENT_TRUST_THRESHOLD sensitivity by lowering the threshold, or raise MIN_BOUNDARY_OUTLIER_RATIO_TO_TRIM.")
print("- If obvious boundary noise remains: lower BOUNDARY_TRIM_DEPTH_THRESHOLD or MIN_BOUNDARY_OUTLIER_RATIO_TO_TRIM.")
print("- If too many reviews: raise SPLIT_REVIEW_SCORE_THRESHOLD or HIGH_REFINE_RISK_THRESHOLD.")
print("- If depth seems inverted: toggle DEPTH_INVERT.")
print("- If transparent/reflective objects fail: keep depth as weak signal only.")

## 11. Important Constraints

- This is a small notebook experiment.
- It does not change the main training crop pipeline.
- It does not implement tracking, Qwen/VLM review, full 3D maps, or production export.
- Depth refinement is conservative: uncertain cases keep the original SAM2 mask and become review signals.